---
# STEP 5 FPI 산출 및 민감도 분석 (수정본)

## 수정 사항

기존 FPI 계산에서 프랜차이즈의 `tfidf_sentiment`(감성점수)가 음수인 경우
FPI 전체가 음수로 산출되는 문제가 발견되었다.

**문제 원인**:
FPI 분자 = 별점(양수) × ln(리뷰수+1)(양수) × 감성점수(음수 가능)
→ 감성점수만 음수가 될 수 있어 FPI 부호를 감성점수가 단독으로 결정하는 구조

**해결 방법: Shift 정규화 적용**
감성점수에서 최솟값을 빼서 전체를 양수 범위로 이동시킨다.

| 방법 | 수식 | 특징 |
|---|---|---|
| 기존 | sentiment 그대로 사용 | 음수 FPI 발생 |
| MinMax | (x - min) / (max - min) | 0~1로 압축, 절대적 크기 손실 |
| **Shift (채택)** | **x - min** | 음수 제거 + 상대적 간격 보존 + 절대적 크기 유지 |

Shift 정규화는 값들 간의 절대적 간격을 보존하면서 음수만 제거한다.
최솟값을 가진 프랜차이즈만 감성점수=0이 되고, 나머지는 원래 간격 그대로 유지된다.

## 공통 라이브러리 및 설정

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

PATH_to_data = "../results"
PATH_to_save = "../results"

print("라이브러리 로드 완료")

라이브러리 로드 완료


## 데이터 준비

In [2]:
df = pd.read_csv(f"{PATH_to_data}/biz_sentiment.csv")

indie_df = df[df['is_franchise'] == False].copy()
fran_df  = df[df['is_franchise'] == True].copy()

print(f"독립 브랜드: {len(indie_df):,}개")
print(f"프랜차이즈 : {len(fran_df):,}개")
print(f"\n프랜차이즈 감성점수 분포:")
print(fran_df['tfidf_sentiment'].describe().round(4))
print(f"\n음수 감성점수 프랜차이즈: {(fran_df['tfidf_sentiment'] < 0).sum()}개")

독립 브랜드: 4,818개
프랜차이즈 : 1,081개

프랜차이즈 감성점수 분포:
count    1081.0000
mean       -0.0086
std         0.0152
min        -0.1114
25%        -0.0185
50%        -0.0084
75%         0.0011
max         0.0448
Name: tfidf_sentiment, dtype: float64

음수 감성점수 프랜차이즈: 782개


---
## STEP 5-0. 감성점수 Shift 정규화

기존 감성점수의 최솟값을 0으로 이동시켜 전체를 양수 범위로 변환한다.
값들 간의 상대적 간격과 절대적 크기는 그대로 유지된다.

In [3]:
# Shift 정규화: 최솟값을 빼서 전체를 양수로 이동
sentiment_min = fran_df['tfidf_sentiment'].min()
fran_df['sentiment_shifted'] = fran_df['tfidf_sentiment'] - sentiment_min

print(f"감성점수 최솟값 (shift 기준): {sentiment_min:.4f}")
print(f"\nShift 정규화 전후 비교:")
print(f"{'':15} {'최솟값':>10} {'최댓값':>10} {'평균':>10}")
print(f"{'정규화 전':15} {fran_df['tfidf_sentiment'].min():>10.4f} "
      f"{fran_df['tfidf_sentiment'].max():>10.4f} "
      f"{fran_df['tfidf_sentiment'].mean():>10.4f}")
print(f"{'정규화 후':15} {fran_df['sentiment_shifted'].min():>10.4f} "
      f"{fran_df['sentiment_shifted'].max():>10.4f} "
      f"{fran_df['sentiment_shifted'].mean():>10.4f}")
print(f"\n음수값 잔존 여부: {(fran_df['sentiment_shifted'] < 0).sum()}개")

감성점수 최솟값 (shift 기준): -0.1114

Shift 정규화 전후 비교:
                       최솟값        최댓값         평균
정규화 전              -0.1114     0.0448    -0.0086
정규화 후               0.0000     0.1561     0.1028

음수값 잔존 여부: 0개


---
## STEP 5-1. 거리 계산

모든 독립 브랜드와 프랜차이즈 간의 물리적 거리를 하버사인 공식으로 계산한다.
행렬 연산으로 고속 처리한다.

In [4]:
print("독립 브랜드 - 프랜차이즈 간 거리 계산 중...")

indie_lat = np.radians(indie_df['latitude'].values)[:, np.newaxis]
indie_lon = np.radians(indie_df['longitude'].values)[:, np.newaxis]
fran_lat  = np.radians(fran_df['latitude'].values)[np.newaxis, :]
fran_lon  = np.radians(fran_df['longitude'].values)[np.newaxis, :]

dlat = indie_lat - fran_lat
dlon = indie_lon - fran_lon
a = np.sin(dlat/2)**2 + np.cos(indie_lat) * np.cos(fran_lat) * np.sin(dlon/2)**2
distances_m = 2 * np.arcsin(np.sqrt(a)) * 6371000

print(f"거리 행렬 shape: {distances_m.shape}")
print(f"  ({len(indie_df):,}개 독립 브랜드 × {len(fran_df):,}개 프랜차이즈)")

독립 브랜드 - 프랜차이즈 간 거리 계산 중...
거리 행렬 shape: (4818, 1081)
  (4,818개 독립 브랜드 × 1,081개 프랜차이즈)


---
## STEP 5-2. FPI 산출 (Shift 정규화 감성점수 적용)

$$FPI = \sum \frac{(\text{별점} \times \ln(\text{리뷰수}+1) \times \text{감성점수}_{shifted})}{(\text{거리(km)}+k)^2}$$

- 감성점수: Shift 정규화 적용 → 항상 0 이상
- 거리: km 단위 변환 (m → km)
- 보정 상수 k=1: 극단적으로 가까운 거리에서 발산 방지
- 반경: 100m / 200m / 300m / 500m 4개 기준으로 산출

In [5]:
# Shift 정규화된 감성점수로 분자 계산
fran_scores = (fran_df['stars'].values
               * np.log(fran_df['review_count'].values + 1)
               * fran_df['sentiment_shifted'].values)

print(f"프랜차이즈 분자 점수 확인:")
print(f"  최솟값: {fran_scores.min():.4f}")
print(f"  최댓값: {fran_scores.max():.4f}")
print(f"  음수 여부: {(fran_scores < 0).sum()}개")

radii = [100, 200, 300, 500]
k = 1  # 거리 보정 상수 (km 단위)
distances_km = distances_m / 1000.0

print(f"\n반경별 FPI 계산 중...")
for r in radii:
    mask = distances_m <= r
    fpi_terms = fran_scores[np.newaxis, :] / (distances_km + k)**2
    fpi_sums  = np.sum(np.where(mask, fpi_terms, 0.0), axis=1)
    indie_df[f'fpi_{r}m'] = fpi_sums
    print(f"  fpi_{r}m 완료 | 음수: {(fpi_sums < 0).sum()}개 | 0초과: {(fpi_sums > 0).sum()}개")

프랜차이즈 분자 점수 확인:
  최솟값: 0.0000
  최댓값: 3.5090
  음수 여부: 0개

반경별 FPI 계산 중...
  fpi_100m 완료 | 음수: 0개 | 0초과: 2013개
  fpi_200m 완료 | 음수: 0개 | 0초과: 3254개
  fpi_300m 완료 | 음수: 0개 | 0초과: 3841개
  fpi_500m 완료 | 음수: 0개 | 0초과: 4330개


---
## STEP 5-3. 원본 데이터에 결합 및 저장

In [7]:
final_df = pd.merge(
    df,
    indie_df[['business_id'] + [f'fpi_{r}m' for r in radii]],
    on='business_id', how='left'
)
for r in radii:
    final_df[f'fpi_{r}m'] = final_df[f'fpi_{r}m'].fillna(0)

final_df.to_csv(f"{PATH_to_save}/biz_sentiment_with_fpi.csv",
                index=False, encoding='utf-8-sig')

print(f"저장 완료: biz_sentiment_with_fpi.csv")
print(f"shape: {final_df.shape}")
print(f"\nFPI 음수 잔존 확인:")
for r in radii:
    neg = (final_df[f'fpi_{r}m'] < 0).sum()
    print(f"  fpi_{r}m 음수: {neg}개")

저장 완료: biz_sentiment_with_fpi.csv
shape: (5899, 22)

FPI 음수 잔존 확인:
  fpi_100m 음수: 0개
  fpi_200m 음수: 0개
  fpi_300m 음수: 0개
  fpi_500m 음수: 0개


---
## STEP 5-4. 민감도 분석

반경별(100m~500m) FPI가 독립 브랜드의 별점과 감성점수에 미치는 영향을
단순 회귀분석으로 비교하여 최적 임계거리를 도출한다.

In [8]:
indie_fpi = final_df[final_df['is_franchise'] == False].copy()

for label, y_var in [("별점(Stars)", "stars"), ("감성점수(Sentiment)", "tfidf_sentiment")]:
    print("\n" + "="*60)
    print(f"  [민감도 분석] 독립 매장 {label}에 미치는 영향")
    print("="*60)
    rows = []
    for r in radii:
        X = sm.add_constant(indie_fpi[f'fpi_{r}m'])
        y = indie_fpi[y_var]
        model = sm.OLS(y, X).fit()
        rows.append({
            '반경': f'{r}m',
            '회귀계수': round(model.params.iloc[1], 4),
            'P-value': round(model.pvalues.iloc[1], 4),
            'R²': round(model.rsquared, 4)
        })
    print(pd.DataFrame(rows).to_string(index=False))


  [민감도 분석] 독립 매장 별점(Stars)에 미치는 영향
  반경    회귀계수  P-value     R²
100m -0.0383   0.0001 0.0034
200m -0.0214   0.0002 0.0029
300m -0.0242   0.0000 0.0072
500m -0.0217   0.0000 0.0116

  [민감도 분석] 독립 매장 감성점수(Sentiment)에 미치는 영향
  반경    회귀계수  P-value     R²
100m -0.0002   0.2672 0.0003
200m -0.0001   0.5950 0.0001
300m -0.0002   0.0207 0.0011
500m -0.0002   0.0017 0.0021


---
## STEP 5-5. FPI 구간 분류

In [11]:
indie_fpi = final_df[final_df['is_franchise'] == False].copy()

fpi_nonzero = indie_fpi[indie_fpi['fpi_300m'] > 0]['fpi_300m']
median_fpi  = fpi_nonzero.median()

def assign_group(fpi):
    if fpi == 0:
        return 'NP'
    elif fpi <= median_fpi:
        return 'LP'
    else:
        return 'HP'

indie_fpi['fpi_group'] = indie_fpi['fpi_300m'].apply(assign_group)

print("FPI 구간 분포:")
print(indie_fpi['fpi_group'].value_counts())
print(f"\nNaN 여부: {indie_fpi['fpi_group'].isna().sum()}개")
print(f"\n중앙값 기준 (fpi_300m > 0): {median_fpi:.4f}")

indie_fpi.to_csv(f"{PATH_to_save}/biz_indie_with_groups.csv",
                 index=False, encoding='utf-8-sig')
print(f"\n저장 완료: biz_indie_with_groups.csv")

FPI 구간 분포:
fpi_group
LP    1921
HP    1920
NP     977
Name: count, dtype: int64

NaN 여부: 0개

중앙값 기준 (fpi_300m > 0): 2.0791

저장 완료: biz_indie_with_groups.csv
